# ice9 free tier

This notebook walks through the free tier results. Run the first cell once to submit your image, then start with the image-level moderation fields before dropping down to service output.

**Before you start:**
- Set your API key: `export ICE9_API_KEY=ice9_...` in your terminal before launching Jupyter, or set it in the cell below.
- Install dependencies: `pip install ice9`

In [ ]:
from ice9 import Ice9
from ice9.exceptions import PartialResultError

# Set your image path here
IMAGE = "path/to/your/image.jpg"

# If you didn't set ICE9_API_KEY in your environment, you can set it here instead:
# import os
# os.environ["ICE9_API_KEY"] = "ice9_..."

client = Ice9()

try:
    result = client.analyze(IMAGE)
except PartialResultError as e:
    print(f"Warning: some services failed: {e.result.services_failed}")
    result = e.result

print(f"Done. Image ID: {result.image_id}")
print(f"NSFW: {result.is_nsfw}")
print(f"Reason: {result.moderation.reason}")
if result.scene:
    print(f"Scene: {result.scene.type} / {result.scene.intimacy}")

## Moderation summary

Start with the image-level moderation result. `is_nsfw` is the quick decision, `moderation.reason` is the explanation, and `scene` carries the structured scene summary.

In [ ]:
print(f"NSFW: {result.is_nsfw}")
print(f"Reason: {result.moderation.reason}")
if result.scene:
    print(f"Scene type: {result.scene.type}")
    print(f"Intimacy:  {result.scene.intimacy}")
    print(f"Activity:  {result.scene.activity or '—'}")

If you want the underlying evidence, you can still inspect the flagged detections directly:

In [ ]:
for detection in result.nsfw_detections():
    print(f"{detection['label']}  confidence={detection['confidence']:.0%}")

## Scene details

The `scene` summary is derived from `content_analysis` and gives you the structured moderation view without needing to parse the service payload yourself.

In [ ]:
if result.scene:
    print(f"Scene type:      {result.scene.type}")
    print(f"Intimacy:       {result.scene.intimacy}")
    print(f"Activity:       {result.scene.activity or '—'}")
    print(f"Activities:     {result.scene.activities}")
    print(f"Anatomy exposed:{result.scene.anatomy_exposed}")
else:
    print("No scene summary.")

In [ ]:
# Gender breakdown — confidence and vote details
if result.content_analysis is not None:
    full_analysis = result.content_analysis.full_analysis
    print(full_analysis.get("gender_breakdown"))

In [ ]:
# Full analysis — all the detail if you need it
import json
if result.content_analysis is not None:
    print(json.dumps(result.content_analysis.full_analysis, indent=2, default=str))

## Colors

The dominant colors in the image, as hex codes.

In [ ]:
if result.colors is not None:
    print(result.colors.dominant)

## Metadata

File format, dimensions, and EXIF data.

In [ ]:
if result.metadata is not None:
    for key, value in result.metadata._data.items():
        print(f"{key}: {value}")

## OCR — text in the image

In [ ]:
if result.ocr is not None:
    text = result.ocr._data.get("text") or ""
    if text.strip():
        print(text)
    else:
        print("No text found.")

## QR codes and barcodes

In [ ]:
if result.qr is not None:
    codes = result.qr._data.get("codes") or []
    if codes:
        for code in codes:
            print(f"[{code.get('type', '?')}] {code.get('data', '')}")
    else:
        print("No codes found.")

## Censoring the image

If the image was flagged, you can draw over the flagged regions using `result.moderation.censor()`.

Requires Pillow: `pip install Pillow`

In [ ]:
from IPython.display import display

censored = result.moderation.censor(IMAGE, method="pixelate")
display(censored)

## Full result as JSON

Everything in one dict, useful for saving to a file or sending to another service.

In [ ]:
print(result.to_json(indent=2))